# Train the flower classifier on a free Colab GPU

This notebook trains the model from this repository on Google Colab's free GPU. It downloads the dataset,
trains the classifier and fine-tunes it, then shows the learning curves and the evaluation report. It
takes roughly 30-60 minutes on a free T4 GPU.

**Before you start:** *Runtime -> Change runtime type -> Hardware accelerator: GPU (T4)*, then *Runtime -> Run all*.

In [ ]:
# Check that a GPU is available
import torch
print('PyTorch', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none - switch the runtime to GPU!')

## Optional: keep results on Google Drive

Colab deletes files when the session ends and can disconnect during long runs. Set `USE_DRIVE = True` to save
checkpoints to Google Drive. If the session disconnects, run the notebook again and training resumes where it
stopped.

In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/flower-classifier'
else:
    SAVE_DIR = '/content/checkpoints'
print('Saving results to', SAVE_DIR)

## Get the code and the data

In [ ]:
%cd /content
!test -d Image-Classifier || git clone https://github.com/thomasmd321/Image-Classifier.git
%cd /content/Image-Classifier
!git pull --quiet
!pip install --quiet -r requirements.txt
!python download_data.py

## Train

Phase 1 trains the new classifier; phase 2 fine-tunes the last block of the pretrained network.
Change the settings below to experiment, e.g. `ARCH = 'efficientnet_b0'`.

In [ ]:
import os

ARCH = 'densenet121'
EPOCHS = 20            # classifier phase (stops early when validation accuracy stalls)
FINETUNE_EPOCHS = 5    # fine-tuning phase
SEED = 42

os.makedirs(SAVE_DIR, exist_ok=True)
last = os.path.join(SAVE_DIR, 'last_checkpoint.pt')
resume = '--resume {}'.format(last) if os.path.exists(last) else ''
if resume:
    print('Resuming from', last)

!python train.py flowers --gpu --arch {ARCH} --epochs {EPOCHS} --finetune_epochs {FINETUNE_EPOCHS} \
    --seed {SEED} --num_workers 2 --save_dir {SAVE_DIR} {resume}

## Learning curves

In [ ]:
from IPython.display import Image, display
display(Image(os.path.join(SAVE_DIR, 'history.png')))

## Evaluation report

In [ ]:
!python evaluate.py flowers {SAVE_DIR}/check_point.pt --category_names cat_to_name.json --gpu \
    --tta --output_dir {SAVE_DIR}/report --num_workers 2
display(Image(os.path.join(SAVE_DIR, 'report', 'confusion_matrix.png'), width=900))
display(Image(os.path.join(SAVE_DIR, 'report', 'misclassified.png'), width=900))

## Try a prediction

In [ ]:
!python predict.py flowers/test/10/image_07104.jpg {SAVE_DIR}/check_point.pt --category_names cat_to_name.json \
    --top_k 5 --gpu --plot_dir {SAVE_DIR}/plots
display(Image(os.path.join(SAVE_DIR, 'plots', 'image_07104.png'), width=450))

## Download the results

Downloads a zip with the best checkpoint, the training history and the evaluation report. Put
`check_point.pt` next to `app.py` to run the web demo, and copy the numbers into the README's results table.

In [ ]:
import shutil
from google.colab import files

bundle = shutil.make_archive('/content/flower-classifier-results', 'zip', SAVE_DIR)
files.download(bundle)